# 6. Release KEGG-BGE on the Hugging Face Hub

Converts the trained checkpoint (`best/`) to the **sentence-transformers format** and uploads it to the Hugging Face Hub, after which anyone can use it with two lines:

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('<account>/<model-name>')
```

Steps: ① convert → ② **check that it reproduces the embeddings used in the paper** → ③ write the model card → ④ upload (private at first) → ⑤ download from the Hub and check again.
A CPU runtime is enough. You need a Hugging Face account and an access token with **write** permission (Settings → Access Tokens).

## 1. Settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U sentence-transformers huggingface_hub

import glob, json
from pathlib import Path
BASE = Path('/content/drive/MyDrive/kegg-bge')     # same Drive folder as notebooks 1–4

HF_REPO = 'YOUR_HF_ID/kegg-bge-base-en-v1.5'    # <- your account or organization
PRIVATE = True                                   # keep private during review; make public from the Hub settings later

runs  = json.load(open(BASE / 'bienc_runs.json'))                 # written by notebook 3
MAIN  = Path(runs['kegg_bge_s42']) / 'best'                        # KEGG-BGE in the paper (seed 42)
SEEDS = [runs['kegg_bge_s1'], runs['kegg_bge_s2']]                # seed 1, 2 (uploaded as branches)
UPLOAD_SEEDS = True

OUT = Path('/content/release')
print(MAIN, MAIN.exists()); print(SEEDS)
for s in SEEDS:   # show the seed recorded in each run folder, if any
    for f in ('config.json', 'args.json', 'best_info.json'):
        p = Path(s) / f
        if p.exists():
            print(s, f, open(p).read()[:200].replace('\n', ' '))

## 2. Convert to sentence-transformers

Same as in training and evaluation: **[CLS] token → L2 normalization**, at most 512 tokens, **abstract only** as input (no BGE query instruction).

In [ ]:
from sentence_transformers import SentenceTransformer, models

def to_sentence_transformer(ckpt, out_dir):
    word = models.Transformer(str(ckpt), max_seq_length=512)
    pool = models.Pooling(word.get_word_embedding_dimension(), pooling_mode='cls')
    st = SentenceTransformer(modules=[word, pool, models.Normalize()])
    st.save(str(out_dir))
    return st

st = to_sentence_transformer(MAIN, OUT / 'main')
print(st)

## 3. Check: same embeddings as in the paper

Compares with the KEGG test embeddings saved during evaluation (`embeddings/test/kegg_bge_s42__cls.npy`). Evaluation used bf16, so the vectors are not bit-identical, but the **cosine similarity should be above 0.999**. Pathway-level mAP is recomputed as well and should be close to 0.764.

In [ ]:
import numpy as np, torch
from sklearn.metrics import average_precision_score

emb_dir = BASE / 'embeddings/test'
pmids = json.load(open(emb_dir / 'pmids.json'))
pathways = json.load(open(BASE / 'data/splits/pathways_test.json', encoding='utf-8'))
abstract, labels = {}, {}
for e in pathways:
    for r in e['references']:
        abstract[r['pmid']] = r['abstract']
        labels.setdefault(r['pmid'], set()).add(e['pathway_id'])

E_new = st.encode([abstract[p] for p in pmids], batch_size=64, convert_to_numpy=True, show_progress_bar=True,
                  device='cuda' if torch.cuda.is_available() else 'cpu')
E_old = np.load(emb_dir / 'kegg_bge_s42__cls.npy')
E_old = E_old / np.linalg.norm(E_old, axis=1, keepdims=True)
cos = (E_new * E_old).sum(1)
print(f'cosine(new, paper): min {cos.min():.5f}  mean {cos.mean():.5f}')

def pathway_map(E):
    S = E @ E.T
    np.fill_diagonal(S, -np.inf)
    aps = []
    for i, p in enumerate(pmids):
        rel = np.array([bool(labels[p] & labels[q]) for q in pmids]); rel[i] = False
        if rel.any():
            m = np.ones(len(pmids), bool); m[i] = False
            aps.append(average_precision_score(rel[m], S[i, m]))
    return np.mean(aps)
print(f'pathway mAP  new {pathway_map(E_new):.4f}  paper {pathway_map(E_old):.4f}')
assert cos.min() > 0.99, 'Embeddings differ: check MAIN and the pooling.'

## 4. Model card (README.md)

This is the page shown on the Hub. Add the paper title and DOI after publication; you can also edit the text in the cell directly.

In [ ]:
CARD = '''---
language: en
license: mit
library_name: sentence-transformers
pipeline_tag: sentence-similarity
base_model: BAAI/bge-base-en-v1.5
tags:
- sentence-transformers
- feature-extraction
- biomedical
- scientific-documents
- kegg
- pathway
---

# KEGG-BGE: pathway-supervised document embeddings for biomedical abstracts

This model embeds a biomedical abstract so that papers studying the same **biological context** (the same KEGG pathway, or nearby pathways in the KEGG hierarchy) are close in the embedding space.
It is [BAAI/bge-base-en-v1.5](https://huggingface.co/BAAI/bge-base-en-v1.5) fine-tuned with a contrastive objective in which papers cited by the same KEGG pathway map are positives.

## Usage

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("REPO_ID")
emb = model.encode(["<abstract 1>", "<abstract 2>"])   # L2-normalized, 768-d
sim = emb @ emb.T                                         # cosine similarity
```

- Input: the **abstract only**, without a query instruction (as in training). Texts longer than 512 tokens are truncated.
- Pooling: [CLS] token followed by L2 normalization (same as BGE).

## Training

- Data: references of KEGG reference pathway maps (KEGG Release 119.0; six categories, Drug Development excluded); 320 training pathways, 4,359 papers. Validation and test pathways are disjoint from training.
- Objective: InfoNCE (temperature 0.05) with in-batch negatives, papers from the same pathway masked out of the negatives, plus hierarchy-graded explicit negatives.
- Learning rate 5e-6, batch size 32, checkpoint selected on validation pathway-level mAP (step 250), seed 42.
- Other seeds (1, 2) are in the `seed1` and `seed2` branches: `SentenceTransformer("REPO_ID", revision="seed1")`.

## Evaluation (test pathways unseen in training)

| | Pathway mAP | Subcategory mAP | Pathway silhouette | Pair AUROC |
|---|---|---|---|---|
| BGE-base-en-v1.5 | 0.718 | 0.563 | 0.165 | 0.929 |
| KEGG-BGE (mean ± SD, 3 seeds) | 0.762 ± 0.002 | 0.602 ± 0.002 | 0.248 ± 0.002 | 0.955 ± 0.001 |

Without further training, the model also improved retrieval over BGE on Reactome (all three levels) and on MeSH (Levels 2 to 4).
See the paper for details and confidence intervals.

## Limitations

- Trained on KEGG reference-pathway literature; the finest distinctions it learns are specific to the KEGG hierarchy and transferred only partly to Reactome and MeSH.
- No gain at the broadest (category) level. Not evaluated for query-to-document search, full texts or non-English text.

## Citation

Paper under review. Citation will be added here.
'''.replace('REPO_ID', HF_REPO)
for d in [OUT / 'main']:
    (d / 'README.md').write_text(CARD, encoding='utf-8')
print(CARD[:600])

## 5. Upload

Asks for the token on the first run (write permission). Private by default (`PRIVATE=True`). Seeds 1 and 2 go to the `seed1` and `seed2` branches of the same repository.

In [ ]:
from huggingface_hub import login, HfApi
login()   # paste the token
api = HfApi()
api.create_repo(HF_REPO, repo_type='model', private=PRIVATE, exist_ok=True)
api.upload_folder(repo_id=HF_REPO, folder_path=str(OUT / 'main'), commit_message='KEGG-BGE (seed 42), model used in the paper')

if UPLOAD_SEEDS:
    for i, s in enumerate(SEEDS, start=1):           # SEEDS = [seed 1 run, seed 2 run]
        branch = f'seed{i}'
        d = OUT / branch
        to_sentence_transformer(Path(s) / 'best', d)
        (d / 'README.md').write_text(CARD, encoding='utf-8')
        api.create_branch(HF_REPO, branch=branch, exist_ok=True)
        api.upload_folder(repo_id=HF_REPO, folder_path=str(d), revision=branch, commit_message=f'KEGG-BGE {branch}')
print('https://huggingface.co/' + HF_REPO)

## 6. Download from the Hub and check

In [ ]:
import shutil
m = SentenceTransformer(HF_REPO)   # a private repository needs the login above
E_hub = m.encode([abstract[p] for p in pmids[:200]], batch_size=64)
print('cosine vs local:', float((E_hub * E_new[:200]).sum(1).min()))

## 7. Before making it public

- At publication (or preprint), set the Hub repository to **Settings → Visibility → Public**.
- On the same page, **Generate DOI** gives the model a DOI to cite in the Data availability statement.
- Replace the Citation section of the model card with the paper.
- For a double-blind journal, keep the repository private during review and write "available upon publication" in the manuscript.